In [ ]:

!pip install --force-reinstall pytorch-lightning==1.9.5
!pip install --force-reinstall torchmetrics==0.11.2 
!pip install --force-reinstall torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install urllib3==1.26


In [2]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = "false"

In [3]:
import re
import pandas as pd
import numpy as np
import yaml
from yaml import CLoader
import os
from os.path import join
from tqdm import tqdm
import re
from tokenizers import Tokenizer
from pyarabic.trans import normalize_digits

In [4]:
class TokenHandler:
    def __init__(self, json_path: str, lang='en'):

        self.tok = Tokenizer.from_file(json_path)
        self.tok.enable_padding(pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
        if lang == 'en':
            self.pre = self.english_preprocess
        elif lang == 'ar':
            self.pre = self.arabic_preprocess
        else:
            raise NotImplementedError('This class suports En and Ar language only for now')
    
    def arabic_preprocess(self, s: str):
        '''Remove non arabic characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''
        s = normalize_digits(s, source='all', out='west')
        s = re.sub(r'[?]+', "؟", s)
        s = re.sub('[" "]+', ' ', s)
        s = re.sub('[^\sء-ي؟:()!,،.:1-9]+', '', s)
        return s
    
    def english_preprocess(self, s: str):
        '''Remove non english characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''

        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^\sa-zA-Z0-9?:()!,،'.:]+", "", s)

        return s
        
    
    def __call__(self, text, length=None):
        text = self.pre(text)
        out = self.tok.encode(text)
        if length is not None:
            out.pad(length, pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
            out.truncate(length)            
        return out.ids
        
    def encode(self, text: str):
        '''@input: text --> single string.
        @return:  ids, tokens
        '''
        text = self.pre(text)
        out = self.tok.encode(text)
        return out
    
    def get_id(self, token: int):
        '''@input: token --> single word 
        @return: id --> int
        '''
        return self.tok.token_to_id(token)
    
    def encode_batch(self, data: list):
        '''@input: data --> list of strings.
        @return:  ids, tokens
        '''
        data = tuple(map(self.pre, data))
        output = self.tok.encode_batch(data)
        return output
    
    def decode(self, ids: list):
        '''@input: ids --> list of int
        @return: text --> single string.
        '''
        return self.tok.decode(ids)
    
    def decode_batch(self, ids: list):
        return self.tok.decode_batch(ids)


In [5]:
en_json = '/kaggle/input/helper-for-s2t/tokenizers/en_tokenizer.json'
ar_json = '/kaggle/input/helper-for-s2t/tokenizers/ar_tokenizer.json'

In [6]:
import torch
import pytorch_lightning as pl

import numpy as np
import os
from torch.utils.data import IterableDataset, DataLoader, Dataset
import gc
gc.collect()

/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZTVN10tenso

2787

In [7]:
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass, field

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from pytorch_lightning.callbacks import Callback, RichProgressBar, ModelCheckpoint, StochasticWeightAveraging
from pytorch_lightning.callbacks.progress.rich_progress import RichProgressBarTheme

import torchmetrics.functional as MF

from rich import print as rprint

In [ ]:
class MuSTCDataset(Dataset):
    def __init__(self, data, config):
        super(MuSTCDataset, self).__init__()
        self.files = []
        for f in data:
            in_dire = [i for i in os.listdir(f) if '.npy' in i]
            in_dire = list(map(lambda x: os.path.join(f, x), in_dire))
            self.files.extend(in_dire)
        self.files = np.array(self.files)
        self.config = config

            
    def  __getitem__(self, idx):
        data = self.files[idx]
        data = np.load(data, allow_pickle=True)
        wavex = data[:, :240000].astype(np.float32)[:, np.newaxis, :]
        en    = data[:, 240000:240060].astype(np.int64)
        ar    = data[:, 240100:240160].astype(np.int64)

        return wavex, en, ar
        
    def __len__(self):
        return len(self.files)

In [ ]:
class Wave2Chunk(nn.Module):
    
    def __init__(self, **kwargs):
        '''frame_size, frame_stride, b1, b2, b3, b4, out_dim=512'''
        
        super(Wave2Chunk, self).__init__()
        
        self.config = kwargs
        assert self.config['frame_size'] >= 2000, "the frame size minimum should be greater than or equal 2000"
        assert not self.config['frame_size'] % self.config['b4'], 'frame size should be divisible by num of fiinal chanels'
        
        pram_per_layer = [(11, 3, 0, 3), (11, 3, 0, 2), (2, 2, 0, 1),
                          (7, 1, 0, 1), (7, 2, 0, 1), (3, 3, 0, 1),
                          (5, 2, 0, 1), (5, 2, 0, 1), (2, 1, 0, 1),   
                          (3, 1, 0, 1), (3, 2, 0, 1), (2, 2, 0, 1)]
        
        out_shape = self.config['frame_size']
        for p in pram_per_layer:
            out_shape = self._conv_output_length(out_shape, *p)
        
        self.conv_block_1 = nn.Sequential(nn.Conv1d(1, self.config['b1'] // 2, 11, 3, dilation=3),  
                                          nn.ReLU(),
                                          nn.Conv1d(self.config['b1'] // 2, self.config['b1'], 11, 3, dilation=2),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 2))
        
               
        self.conv_block_2 = nn.Sequential(nn.Conv1d(self.config['b1'], self.config['b2'] // 2, 7, 1, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(self.config['b2'] // 2, self.config['b2'], 7, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(3, 3),                  
                                          nn.BatchNorm1d(self.config['b2']))
        
               
        self.conv_block_3 = nn.Sequential(nn.Conv1d(self.config['b2'], self.config['b3'] // 2, 5, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(self.config['b3'] // 2, self.config['b3'], 5, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 1))
        
               
        self.conv_block_4 = nn.Sequential(nn.Conv1d(self.config['b3'], self.config['b4'] // 2, 3, 1, dilation=1),  
                                          nn.ReLU(),
                                          nn.Conv1d(self.config['b4'] // 2, self.config['b4'], 3, 2, dilation=1),  
                                          nn.ReLU(),
                                          nn.MaxPool1d(2, 2),                   
                                          nn.BatchNorm1d(self.config['b4']))
        
        
        
        self.projection = nn.Sequential(nn.Linear(out_shape, self.config['out_dim']//2),
                                        nn.Dropout(0.01),
                                        nn.ReLU(),
                                        nn.Linear(self.config['out_dim']//2, self.config['out_dim']))
        
        self.norm = nn.BatchNorm1d(self.config['out_dim'])
        
    def _conv_output_length(self, length_in, kernel_size, stride=1, padding=0, dilation=1):
        return (length_in + 2 * padding - dilation * (kernel_size - 1) - 1) // stride + 1
    
    def forward(self, waves, mask=False, training=False):
        # ensure that all chunks will be the same size

        waves = self.__pad_wave(waves, mask)
        if mask:
            waves, mask = waves
        else:
            mask = None

#         # convert the the wave into overlabing chunks
        waves = waves.unfold(dimension=2, size=self.config['frame_size'], step=self.config['frame_stride']).transpose(2, 1)
        
        B, chunks, c, seq_len = waves.size()
        
        waves = waves.contiguous().view(B*chunks, c, seq_len)
        
        waves = self.conv_block_1(waves)
        waves = self.conv_block_2(waves)
        waves = self.conv_block_3(waves)
        waves = self.conv_block_4(waves)
        waves = waves.view(-1, waves.size(-1)) #view(B * chunks * c, waves.size(-1))
        
        waves = self.projection(waves)
        waves = self.norm(waves).view(B, -1, waves.size(-1))
        if mask is not None:
            return waves, mask
        return waves
        
        
    def __pad_wave(self, wave, mask=False):
        """Only supported shapes (B, C, Wave_size) or (B, Wave_size) or (Wave_size, ) for unbatched
        """
        
        B, C, w_len = 1, 1, None
        if not isinstance(wave, torch.Tensor):
            wave = torch.tensor(wave)
        wave = wave
        if wave.dim() == 3:
            B, C, w_len = wave.size()
        elif wave.dim() == 2:
            B, w_len = wave.size()
        elif wave.dim() == 1:
            w_len, = wave.size()
            wave = wave.unsqueeze(0)

        else:
            raise ValueError(f'expected number of dims is 1 for unbatched and 2 for batched but you provide {wave.dim()}')
    
        assert w_len >= self.config['frame_size'], f"vrey short wave lenght. minimum wave lenght is {self.config['frame_size']}"
        assert C == 1, f'only support mono wave at this time. Expected num of channels is {1} but given {C}'
        
        wave = wave.reshape(B, C, w_len)                   
        num_frames = int(math.ceil(float(abs(w_len - self.config['frame_size'])) / self.config['frame_stride']))
        plus_len = (num_frames * self.config['frame_stride'] + self.config['frame_size']) - w_len
        wave = F.pad(wave, [0, plus_len], "constant", 0) 
        
        if mask:
            mask = (wave == 0).squeeze(1).to(wave.device)
#             mask = F.pad(mask, [0, plus_len], "constant", True)
            mask = mask.unfold(dimension=1, size=self.config['frame_size'], step=self.config['frame_stride'])
            mask = mask.contiguous().view(B, mask.size(1)*self.config['b4'], -1)
            mask = torch.all(mask, 2)
            return wave, mask
        return wave
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
                
        for m in self.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                m.reset_parameters()
                
    def get_config(self):
        return self.config
    



##########################################

class EncoderLayer(nn.Module):
    def __init__(self, **kwargs):
        """@params:
                    d_model, nhead, nch, dropout=0.3, batch_first=True
        """
        super(EncoderLayer, self).__init__()
        self.config = kwargs        
        
        self.pre_norm = nn.LayerNorm(self.config['d_model'])
        
        self.mha      = nn.MultiheadAttention(embed_dim=self.config['d_model'], num_heads=self.config['nhead'], dropout=0.1, 
                                               bias=True, add_bias_kv=True, batch_first=self.config['batch_first'])
        
        self.res_front = nn.Sequential(nn.Conv1d(1, 3, 7, 1, padding='same'),  
                                       nn.ReLU(),
                                       nn.Conv1d(3, 3, 5, 1, padding='same'),  
                                       nn.Conv1d(3, self.config['nch'], 3, 1, padding='same'),
                                       nn.ReLU())
        
        
        self.res_back  = nn.Linear(self.config['d_model']* self.config['nch'], self.config['d_model'], bias=False)


        self.ffn       = nn.Sequential(nn.LayerNorm(self.config['d_model']),
                                       nn.Linear(self.config['d_model'], self.config['d_model']*2),
                                       nn.ReLU(),
                                       nn.Dropout(0.08),
                                       nn.Linear(self.config['d_model']*2, self.config['d_model']),
                                       nn.ReLU())
        
        
        
    def forward(self, x, padding_mask=None, need_weights=False, training=False):
        if x.dim() < 3:
            x = x.view(1, -1, self.config['d_model']).contiguous()
        x = self.pre_norm(x)
        
        att_out, att_weights = self.mha(query=x, key=x, value=x, key_padding_mask=padding_mask, 
                                        need_weights=need_weights)
        
        att_out = att_out + x
        
        B, T, D = x.size()
        x = x.view(B*T, 1, D)
        x = self.res_front(x)
        
        x = x.view(B, T, self.config['nch']*D)
        x = F.dropout(x, 0.05, training=training)
        x = self.res_back(x)
        
        att_out = att_out + x; del x
        
        att_out = att_out + self.ffn(att_out)
        
        if need_weights:
            return att_out, att_weights
        
        return att_out
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm,)):
                m.reset_parameters()
                
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config
    
    
    
    
class Encoder(nn.Module):
    def __init__(self, **kwargs):
        '''d_model=512, nhead=4, nch=5, dropout=0.3, batch_first=True, size'''
        super().__init__()
        self.config = kwargs
        assert self.config['d_model'] % self.config['nhead'] == 0, 'd_model should be dvisible by num of heads'
        self.enc_stack = nn.ModuleList([EncoderLayer(d_model=self.config['d_model'], nch=self.config['nch'], nhead=self.config['nhead'], 
                                                     dropout=0.1, batch_first=self.config['batch_first']) 
                                        for _ in range(self.config['size'])])
        
        
    def forward(self, x, padding_mask=None, need_weights=False, training=False):
        if need_weights:
            
            for layer in self.enc_stack:
                x, atten_weights = layer(x, padding_mask=padding_mask, need_weights=need_weights, training=training)
            return x, atten_weights
        
        for layer in self.enc_stack:
                x = layer(x, padding_mask=padding_mask, need_weights=need_weights, training=training)
        return x
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm,)):
                m.reset_parameters()
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config
    
    
    
#####################################################################
#####################################################################


class DecoderLayer(nn.Module):
    def __init__(self, **kwargs):
        """@params:
                    d_model, nhead, nch, dropout=0.3, batch_first=True
        """
        super(DecoderLayer, self).__init__()
        self.config = kwargs
        
        self.pre_norm1 = nn.LayerNorm(self.config['d_model'])
        
        self.smha      = nn.MultiheadAttention(embed_dim=self.config['d_model'], num_heads=self.config['nhead'], dropout=0.1, 
                                               bias=True, add_bias_kv=True, batch_first=self.config['batch_first'])
        
        self.res_front = nn.Sequential(nn.Conv1d(1, 3, 7, 1, padding='same'),  
                                       nn.ReLU(),
                                       nn.Conv1d(3, 3, 5, 1, padding='same'),  
                                       nn.Conv1d(3, self.config['nch'], 3, 1, padding='same'),  
                                       nn.ReLU())
        
        self.res_back  = nn.Linear(self.config['d_model']* self.config['nch'], self.config['d_model'], bias=False)


        self.pre_norm2 = nn.LayerNorm(self.config['d_model'])
        
        self.cmha      = nn.MultiheadAttention(embed_dim=self.config['d_model'], num_heads=self.config['nhead']*2, dropout=self.config['dropout'], 
                                               bias=True, add_bias_kv=True, batch_first=self.config['batch_first'])
        
        self.ffn       = nn.Sequential(nn.LayerNorm(self.config['d_model']),
                                       nn.Linear(self.config['d_model'], self.config['d_model']*2),
                                       nn.ReLU(),
                                       nn.Dropout(0.05),
                                       nn.Linear(self.config['d_model']*2, self.config['d_model']),
                                       nn.ReLU())
        
        
    def forward(self, query, key, query_mask=None, key_mask=None, att_mask=None, need_weights=False, training=False):
        if query.dim() < 3:
            query = query.view(1, -1, self.config['d_model']).contiguous()
            
        if key.dim() < 3:
            key = key.view(1, -1, self.config['d_model']).contiguous()
            
        query = self.pre_norm1(query)
        

        
        att_out, _ = self.smha(query, query, query, key_padding_mask=query_mask, need_weights=False, attn_mask=att_mask)
        att_out = att_out + query
        
        B, T, D = query.size()
        query = query.view(B*T, 1, D)
        query = self.res_front(query)
        
        query = query.view(B, T, self.config['nch']*D)
        query = F.dropout(query, 0.06, training=training)

        query = self.res_back(query)
        query = att_out + query; del att_out
        
        
        query = self.pre_norm2(query)
        
        att_out, att_weight = self.cmha(query, key, key, key_padding_mask=key_mask, attn_mask=None,
                                        need_weights=need_weights, )
        
        att_out = att_out + query; del query
        
        att_out = att_out + self.ffn(att_out)
        
        if need_weights:
            return att_out, att_weight
        
        return att_out
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm, )):
                m.reset_parameters()
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config



class Decoder(nn.Module):
    def __init__(self, **kwargs):
        '''d_model=512, nhead=4, nch=5, dropout=0.3, batch_first=True, size'''
        super().__init__()
        self.config = kwargs
        assert self.config['d_model'] % self.config['nhead'] == 0, 'd_model should be dvisible by num of heads'
        
        self.dec_stack = nn.ModuleList([DecoderLayer(d_model=self.config['d_model'], nch=self.config['nch'], nhead=self.config['nhead'], 
                                                     dropout=self.config['dropout'], batch_first=self.config['batch_first']) 
                                        for _ in range(self.config['size'])])
        
        
    def forward(self, query, key, query_mask=None, key_mask=None, att_mask=None, need_weights=False, training=False):
        if need_weights:
            for layer in self.dec_stack:
                query, atten_weight = layer(query, key, query_mask=query_mask, key_mask=key_mask, att_mask=att_mask, 
                                            need_weights=need_weights, training=training)
            return query, atten_weight
        
        for layer in self.dec_stack:
            query = layer(query, key, query_mask=query_mask, key_mask=key_mask, att_mask=att_mask,
                          need_weights=need_weights, training=training)
            
        return query
    
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else:
                nn.init.xavier_normal_(p.data, gain=nn.init.calculate_gain('relu'))
                
        for m in self.modules():
            if isinstance(m, (nn.LayerNorm,)):
                m.reset_parameters()
                
                
    def freeze(self, state=True):
        
        self.requires_grad_(state)
    

                
    def get_config(self):
        return self.config




class Head(nn.Module):
    def __init__(self, **kwargs):
        '''d_model, voc_size'''
        super().__init__()
        self.config = kwargs
        
        self.layer1 =  nn.Sequential(nn.Linear(self.config['d_model'], self.config['voc_size'] // 3),
                                     nn.LeakyReLU(0.1))

        self.layer2 = nn.Sequential(nn.Dropout(0.08),
                                    nn.Linear(self.config['voc_size'] // 3, self.config['voc_size']))

       
    def forward(self, x, **kwargs):
        x = self.layer1(x)
        x = self.layer2(x)        
        return x
        
    def init_weights(self):
        
        for p in self.parameters():
            if p.dim() == 1:
                p.data.zero_()

            else: 
                nn.init.xavier_uniform_(p.data)
                
    def freeze(self, state=True):
        self.requires_grad_(state)

                
    def get_config(self):
        return self.config
    


@dataclass
class HPrams:
    wave_param: dict     = field(default_factory=dict)
    encoder_params: dict = field(default_factory=dict)
    decoder_params: dict = field(default_factory=dict)
    head_names: dict     = field(default_factory=dict)
    head_params: dict    = field(default_factory=dict)
    
    

class Model(nn.Module):  
    
    def __init__(self, wave_param: dict, encoder_params: dict, decoder_params: dict, head_names: dict, head_params: dict):
        super(Model, self).__init__()
        
        self.hparams = HPrams(wave_param=wave_param, encoder_params=encoder_params, decoder_params=decoder_params, 
                              head_names=head_names, head_params=head_params)
        
        self.wave_enc = Wave2Chunk(**self.hparams.wave_param)
        

        self.context_enc = Encoder(**self.hparams.encoder_params)
        
        self.shared_decoder = Decoder(d_model=1080, nhead=60, nch=20, dropout=0.3, batch_first=True, size=3)
        
        self.heads = nn.ModuleDict()
        for h, pad_idx in self.hparams.head_names.items():
            self.heads[h] = nn.ModuleDict({'emp_layer': nn.Sequential(nn.Embedding(num_embeddings=self.hparams.head_params[h]['voc_size'], 
                                                                      embedding_dim=self.hparams.head_params[h]['d_model'], padding_idx=pad_idx),
                                                                      Decoder(**self.hparams.encoder_params, size=2),
                                                                      nn.Liear(self.hparams.)
                                                                     
                                                                     ),
                                   'output_layer': Head(**self.hparams.head_params[h])})
            
    def pe(self, length: int, depth: int, device, n=10000):
        '''create positionalemppeding matrix
        @params:
                length:  Max number of tokens in as sentence that the model will deal with it during inference.
                depth:   Empeddingdim
        '''
        
        positions = torch.arange(length, device=device).view(-1, 1)    # (seq, 1)  [0, 1, 2, 3 ... length-1]

        depths = torch.arange(depth, device=device).view(1, -1) / depth   # (1, depth) [0 / depth, 1 / depth, 2/depth, 3/depth ... length-1/depth]

        angle_rates = 1 / (n**depths)             # (1, depth)

        angle_rads = positions * angle_rates      # (pos, depth)

        angle_rads[:, 0::2] = torch.sin(angle_rads[:, 0::2])

        # apply cos to odd indices in the array; 2i+1
        angle_rads[:, 1::2] = torch.cos(angle_rads[:, 1::2])
    #         print(angle_rads.shape)
        return angle_rads.float()   
    
    def forward(self, wave, target: dict, training=False, wave_mask=False,
                target_mask=False, dec_mask=False, need_dec_weights=False):
        
        model_output = dict() 
        for h, _ in self.hparams.head_names.items():
            model_output[h] = dict()
        
        
        wave = self.wave_enc(wave, mask=wave_mask, training=training)
        
        if wave_mask:
            wave, wave_mask = wave
        else:
            wave_mask = None

        B, T, D = wave.size()
        wave = self.pe(T, D, wave.device) + wave

        wave = self.context_enc(wave, padding_mask=wave_mask, training=training)
    
    
        for h, _ in self.hparams.head_names.items():
            
            target_head = target[h]
            assert target_head.dim() < 3, f'Head: {h}, target size should be 1D tensor for unbatched and 2D for batched'
            if target_head.dim() < 2:
                target_head = target_head.view(1, -1)
                
            query_mask = None
            if target_mask:
                query_mask = (target_head == self.hparams.head_names.get(h)).to(target_head.device)
            
            if dec_mask:
                att_mask = self.look_ahead_mask(target_head.size(1), target_head.size(1), device=target_head.device)
                
            else: 
                att_mask = None
                
            target_head = self.heads[h]['emp_layer'](target_head)
            B, T, D = target_head.size()
            target_head = self.pe(T, D, target_head.device) + target_head
                        
            
            target_head = self.context_dec(query=target_head, key=wave, query_mask=query_mask, key_mask=wave_mask, att_mask=att_mask,
                                              need_weights=need_dec_weights, training=training)

            if need_dec_weights:
                target_head, model_output[h]['attention_weights'] = target_head[0], target_head[1].detach()
            
            model_output[h]['predection'] = self.heads[h]['output_layer'](target_head); del target_head
            
                
        return model_output
    
    def look_ahead_mask(self, tgt_len:int, src_len: int, device):
        mask = torch.triu(torch.ones((tgt_len, src_len), device=device), diagonal=1).type(torch.bool)
        
        return mask
    
    def init_weights(self):
        self.wave_enc.init_weights()
        self.context_enc.init_weights()
        self.context_dec.init_weights()
        for h, _ in self.hparams.head_names.items():
            self.heads[h]['output_layer'].init_weights()
            
            
class Speech2TextArcht(pl.LightningModule):
    def __init__(self, wave_param: dict, encoder_params: dict, decoder_params: dict, head_names: list, head_params: dict, lr: float):
        super(Speech2TextArcht, self).__init__()
        self.save_hyperparameters()
        self.model = Model(wave_param, encoder_params, decoder_params, head_names, head_params)
        
        self.model.init_weights()
#         torch._dynamo.reset()
#         self.model = torch.compile(self.model, dynamic=True)
        
        
    def forward(self, wave, target: dict, training=False, wave_mask=False,
                    target_mask=False, need_dec_weights=False, dec_mask=False):
        
        
        results = self.model(wave, target, training, wave_mask=wave_mask, target_mask=target_mask, 
                             need_dec_weights=need_dec_weights, dec_mask=dec_mask) #dec_mask
        
        return results
    
    def training_step(self, batch, batch_idx):
        
        wave, en, ar = batch
        ground_truth = {'en': en[:, 1:], 'ar': ar[:, 1:]}
        results = self(wave=wave, target={'en': en[:, :-1], 'ar': ar[:, :-1]}, training=True, wave_mask=True, 
                       target_mask=True, dec_mask=True)
        loss = 0
        named_loss = dict()
        named_loss[f'Loss'] = loss
        for h, pad_idx in self.hparams.head_names.items():
            preds = results[h]['predection'].transpose(2, 1)
            h_loss = F.cross_entropy(preds, ground_truth[h], reduction='mean', ignore_index=pad_idx, label_smoothing=0.09)
            loss += h_loss
            
            
            named_loss[f'{h}_Acc'] = MF.classification.multiclass_accuracy(preds.detach(), ground_truth[h], ignore_index=pad_idx,
                                                                           num_classes=self.hparams.head_params[h]['voc_size'])
        named_loss[f'Loss'] = loss.detach()
                
        
        self.log_dict(named_loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        
        return loss
    def validation_step(self, batch, batch_idx):
        at = 'val'
            
        wave, en, ar = batch
        ground_truth = {'en': en[:, 1:], 'ar': ar[:, 1:]}
        results = self(wave=wave, target={'en': en[:, :-1], 'ar': ar[:, :-1]}, training=False, wave_mask=True, 
                       target_mask=True, dec_mask=True)
        loss = 0
        named_loss = dict()
        for h, pad_idx in self.hparams.head_names.items():
            preds = results[h]['predection'].transpose(2, 1)
            h_loss = F.cross_entropy(preds, ground_truth[h], reduction='mean', ignore_index=pad_idx, label_smoothing=0.09)
            loss += h_loss
#             named_loss[f'{h}_{at}_Loss'] = h_loss
            named_loss[f'{h}_{at}_Acc'] = MF.classification.multiclass_accuracy(preds.detach(), ground_truth[h], ignore_index=pad_idx,
                                                                           num_classes=self.hparams.head_params[h]['voc_size'])
        named_loss['val_loss'] = loss.detach()
        self.log_dict(named_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True,)
        
        
        
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, betas=(0.55, 0.6), weight_decay=0.2, eps=1e-7, amsgrad=True, fused=True)
        scheduler = {
            "scheduler": CosineAnnealingWarmRestarts(optimizer, T_0=8, T_mult=2, eta_min=9e-4, verbose=True),
            "interval": "epoch",
            "frequency": 1,}
        return [optimizer], [scheduler]


In [ ]:

class Predictions(Callback):
    def __init__(self, config):
        self.tokenizers = dict()
        for k, v in config.items():
            self.tokenizers[k] = TokenHandler(v, k)
        
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, *args, **kwargs):
        if not batch_idx % 3000:
            wave, en, ar = batch
            ground_truth = {'en': en[:, 1:], 'ar': ar[:, 1:]}
            with torch.no_grad():
                results = pl_module(wave=wave, target={'en': en[:, :-1], 'ar': ar[:, :-1]}, training=False, wave_mask=True, 
                            target_mask=True, dec_mask=True)
            pred = ''
            ground = ''
            r = torch.randint(0, en.size(0), (1, )).item()
            for h, pad_idx in pl_module.hparams.head_names.items():

                t = self.tokenizers[h].decode_batch(results[h]['predection'].detach().argmax(-1).tolist())
                j = self.tokenizers[h].decode_batch(ground_truth[h].detach().tolist())
                blue = MF.bleu_score(t, j)
                
                pred += f"'{h} guess: with blue score: {blue:0.4f} ' \n\t {t[r]}\n\n"
                ground += f"'{h}' \n\t {j[r]}\n\t"

            rprint(f'\nGround Truth {batch_idx}: \n\t {ground} \n\n {pred}')
                
    def load_state_dict(self, state_dict):
        self.tokenizers.update(state_dict)

    def state_dict(self):
        return self.tokenizers.copy()

    
    

# create your own theme!
progress_bar = RichProgressBar(theme=RichProgressBarTheme(description="green_yellow",
                                                          progress_bar="green1",
                                                          progress_bar_finished="green1",
                                                          progress_bar_pulse="#6206E0",
                                                          batch_progress="green_yellow",
                                                          time="grey82",
                                                          processing_speed="grey82",
                                                          metrics="green1",
                                                        ))


ckp = ModelCheckpoint(every_n_train_steps=50, save_last=True, auto_insert_metric_name=False,
                      dirpath='/kaggle/working/lightning_logs/checkpoints')
swa = StochasticWeightAveraging(swa_lrs=1e-2, annealing_epochs=20, swa_epoch_start=4)

In [ ]:
train_path = '/kaggle/input'
train_path = [os.path.join(train_path, i) for i in os.listdir(train_path) if 'spliter' in i]
train_path.append('/kaggle/input/dataframes/data/train')
val_path = ['/kaggle/input/dataframes/data/dev']

pl.seed_everything(22, workers=True)
worker = 2
accelerator = 'auto'
devices = 2
strategy = 'auto'
epochs = 10000

In [ ]:
class DataLightning(pl.LightningDataModule):
    def __init__(self):
        super().__init__()

    def train_dataloader(self):
        train_loader = MuSTCDataset(train_path, wave_param)
        
        train_loader = DataLoader(train_loader, batch_size=None,  shuffle=True, prefetch_factor=2, pin_memory=True, num_workers=worker, persistent_workers=True)
        return train_loader 
    def val_dataloader(self):
        val_loader = MuSTCDataset(val_path, wave_param)
        val_loader = DataLoader(val_loader, batch_size=None,  shuffle=False, prefetch_factor=2, pin_memory=True, num_workers=worker, persistent_workers=True,)
        return val_loader


In [ ]:
pred = Predictions({'ar': ar_json, 'en': en_json})        
wave_param      = dict(frame_size=20000, frame_stride=16000, b1=5, b2=10, b3=15, b4=20, out_dim=512)
datamoel = DataLightning()
encoder_params  = dict(d_model=512, nhead=8, nch=15, dropout=0.3, batch_first=True, size=5)
decoder_params  = dict(d_model=512, nhead=16, nch=20, dropout=0.5, batch_first=True, size=8)

head_params     = dict(en=dict(d_model=512, voc_size=500), ar=dict(d_model=512, voc_size=500))

tokenizers      = dict(en=TokenHandler(en_json, 'en'),
                           ar=TokenHandler(ar_json, 'ar'))

head_names      = dict(en=tokenizers['en'].get_id("<PAD>"), ar=tokenizers['ar'].get_id("<PAD>"))
hyper_parameter = dict(lr=0.2, wave_param=wave_param, encoder_params=encoder_params, decoder_params=decoder_params, 
                        head_names=head_names, head_params=head_params)

model = Speech2TextArcht(**hyper_parameter)

# ckpppp = torch.load('/kaggle/working/m.pth')

# model.load_state_dict(ckpppp)
# !rm -r /kaggle/working/m.pth

In [ ]:
# # import os
# import shutil

# os.makedirs('lightning_logs', exist_ok=True)

# root = '/kaggle/input/model-checkpoint/lightning_logs'
# for d in os.listdir(root):

#     dest = os.path.join('lightning_logs', d)
#     os.makedirs(dest, exist_ok=True)
#     for f in os.listdir(os.path.join(root, d)):
#         shutil.copy2(os.path.join(root, d, f), dest)

In [ ]:
trainer = pl.Trainer(accelerator=accelerator, devices=devices, 
                     max_epochs=epochs,
                     strategy='auto',
                     num_sanity_val_steps=1,
                     log_every_n_steps=400,
                     callbacks=[ ckp, swa, pred],
                     accumulate_grad_batches=65,
#                      limit_train_batches=12000,
                     gradient_clip_val=8,

                     logger=True,
                     enable_model_summary=True, enable_checkpointing=True,# benchmark=True, 
                     default_root_dir='/kaggle/working/')

trainer.fit(model, datamoel,)# ckpt_path='/kaggle/input/model-checkpoint/lightning_logs/checkpoints/last.ckpt')

In [ ]:
# import torch
# c = torch.load('/kaggle/input/model-checkpoint/lightning_logs/checkpoints/last.ckpt')
# cc = dict()
# for k, v in c['state_dict'].items():
#     cc[k] = v.cpu()
# torch.save(cc, 'm.pth')